# Module 6 – Final Data Science Project
## Customer Churn Analysis & Prediction
### Codomax Digital Solutions Internship

**Objective:** Build an end-to-end Data Science project that combines exploratory data analysis, visualization, feature engineering, supervised Machine Learning, model evaluation, and portfolio-ready conclusions.

**Dataset:** IBM Telco Customer Churn sample dataset. It contains 7,043 customer records and 21 columns, with `Churn` as the target variable. The public IBM dataset is available on GitHub and is also listed on Kaggle. citeturn0search5turn0search4

**Important:** This is an educational/portfolio analysis of sample data, not a production customer-risk system.

## 1. Business Problem

Customer churn is an important business problem for subscription-based companies. The goal of this project is to:

- Understand customer characteristics associated with churn.
- Explore contract, tenure, service, and billing patterns.
- Build classification models to predict whether a customer is likely to churn.
- Compare model performance using appropriate evaluation metrics.
- Translate the analysis into practical, data-informed observations.

### Questions
1. What is the overall churn rate?
2. How does churn vary by contract type and tenure?
3. How do monthly charges and services relate to churn?
4. Which Machine Learning model performs better on the held-out test set?
5. Which features contribute most to the Random Forest model?

## 2. Import Libraries

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from sklearn.metrics import roc_auc_score, classification_report, confusion_matrix, RocCurveDisplay

sns.set_theme(style='whitegrid')
pd.set_option('display.max_columns', None)

print('Libraries imported successfully.')

## 3. Load the Dataset

In [ ]:
url = 'https://raw.githubusercontent.com/IBM/telco-customer-churn-on-icp4d/master/data/Telco-Customer-Churn.csv'
df = pd.read_csv(url)

print(f'Rows: {df.shape[0]:,}')
print(f'Columns: {df.shape[1]}')
display(df.head())

## 4. Initial Data Exploration

In [ ]:
print('Column names:')
print(df.columns.tolist())

print('\nData types:')
display(df.dtypes.to_frame('dtype'))

print('\nSummary statistics:')
display(df.describe(include='all').transpose().head(25))

## 5. Data Quality Check

In [ ]:
missing_report = pd.DataFrame({
    'Missing Values': df.isna().sum(),
    'Missing %': (df.isna().mean() * 100).round(2),
})
display(missing_report.sort_values('Missing Values', ascending=False).head(10))
print('Duplicate rows:', df.duplicated().sum())
print('Duplicate customer IDs:', df['customerID'].duplicated().sum())

## 6. Data Cleaning and Feature Engineering

In [ ]:
data = df.copy()

# TotalCharges contains blank strings in some records; convert them to numeric.
data['TotalCharges'] = pd.to_numeric(data['TotalCharges'], errors='coerce')

# Remove duplicate customer records if any.
data = data.drop_duplicates(subset='customerID').copy()

# Convert target to binary.
data['Churn'] = data['Churn'].map({'Yes': 1, 'No': 0})

# Create interpretable derived features.
service_cols = [
    'PhoneService', 'MultipleLines', 'OnlineSecurity', 'OnlineBackup',
    'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies'
]
data['Service_Count'] = sum((data[c] == 'Yes').astype(int) for c in service_cols)
data['Tenure_Group'] = pd.cut(
    data['tenure'],
    bins=[-1, 12, 24, 48, np.inf],
    labels=['0-12 months', '13-24 months', '25-48 months', '49+ months']
)

print('Cleaned shape:', data.shape)
print('Missing TotalCharges:', data['TotalCharges'].isna().sum())
display(data.head())

## 7. Target Distribution

In [ ]:
churn_rate = data['Churn'].mean() * 100
print(f'Overall churn rate: {churn_rate:.2f}%')
display(data['Churn'].value_counts().rename(index={0:'Stayed', 1:'Churned'}))

plt.figure(figsize=(7,5))
sns.countplot(data=data, x='Churn')
plt.title('Customer Churn Distribution')
plt.xlabel('Churn (0 = No, 1 = Yes)')
plt.ylabel('Number of Customers')
plt.show()

## 8. Exploratory Data Analysis – Contract and Tenure

In [ ]:
contract_churn = data.groupby('Contract')['Churn'].mean().sort_values(ascending=False) * 100
display(contract_churn.round(2).to_frame('Churn Rate (%)'))

plt.figure(figsize=(8,5))
sns.barplot(x=contract_churn.index, y=contract_churn.values)
plt.title('Churn Rate by Contract Type')
plt.xlabel('Contract Type')
plt.ylabel('Churn Rate (%)')
plt.xticks(rotation=15)
plt.show()

In [ ]:
tenure_churn = data.groupby('Tenure_Group', observed=True)['Churn'].mean() * 100
display(tenure_churn.to_frame('Churn Rate (%)').round(2))

plt.figure(figsize=(8,5))
sns.barplot(x=tenure_churn.index, y=tenure_churn.values)
plt.title('Churn Rate by Tenure Group')
plt.xlabel('Tenure Group')
plt.ylabel('Churn Rate (%)')
plt.show()

## 9. Exploratory Data Analysis – Charges and Services

In [ ]:
plt.figure(figsize=(9,5))
sns.boxplot(data=data, x='Churn', y='MonthlyCharges')
plt.title('Monthly Charges by Churn Status')
plt.xlabel('Churn (0 = No, 1 = Yes)')
plt.ylabel('Monthly Charges')
plt.show()

In [ ]:
service_churn = data.groupby('Service_Count')['Churn'].mean() * 100
display(service_churn.to_frame('Churn Rate (%)').round(2))

plt.figure(figsize=(8,5))
sns.lineplot(x=service_churn.index, y=service_churn.values, marker='o')
plt.title('Churn Rate by Number of Services')
plt.xlabel('Number of Services')
plt.ylabel('Churn Rate (%)')
plt.show()

## 10. Correlation Analysis

In [ ]:
numeric_cols = ['SeniorCitizen', 'tenure', 'MonthlyCharges', 'TotalCharges', 'Service_Count', 'Churn']
corr = data[numeric_cols].corr()
display(corr.round(2))

plt.figure(figsize=(8,6))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='Blues', linewidths=0.5)
plt.title('Correlation Heatmap')
plt.show()

## 11. Prepare Data for Machine Learning

In [ ]:
# Customer ID is an identifier, so it is excluded from the model.
X = data.drop(columns=['Churn', 'customerID'])
y = data['Churn']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

categorical_features = X_train.select_dtypes(include=['object', 'category']).columns.tolist()
numeric_features = X_train.select_dtypes(include=['int64', 'float64']).columns.tolist()

numeric_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

categorical_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

preprocessor = ColumnTransformer([
    ('num', numeric_pipeline, numeric_features),
    ('cat', categorical_pipeline, categorical_features)
])

print('Training rows:', len(X_train))
print('Testing rows:', len(X_test))
print('Numeric features:', len(numeric_features))
print('Categorical features:', len(categorical_features))

## 12. Train Multiple Classification Models

In [ ]:
models = {
    'Logistic Regression': LogisticRegression(max_iter=2000, class_weight='balanced', random_state=42),
    'Random Forest': RandomForestClassifier(n_estimators=300, class_weight='balanced', random_state=42, n_jobs=-1),
    'Gradient Boosting': GradientBoostingClassifier(random_state=42)
}

model_results = []
trained_models = {}

for name, estimator in models.items():
    pipeline = Pipeline([
        ('preprocessor', preprocessor),
        ('model', estimator)
    ])
    pipeline.fit(X_train, y_train)
    pred = pipeline.predict(X_test)
    proba = pipeline.predict_proba(X_test)[:, 1]

    model_results.append({
        'Model': name,
        'Accuracy': accuracy_score(y_test, pred),
        'Precision': precision_score(y_test, pred),
        'Recall': recall_score(y_test, pred),
        'F1 Score': f1_score(y_test, pred),
        'ROC-AUC': roc_auc_score(y_test, proba)
    })
    trained_models[name] = pipeline

results_df = pd.DataFrame(model_results).sort_values('ROC-AUC', ascending=False)
display(results_df.style.format({c:'{:.3f}' for c in results_df.columns if c != 'Model'}))

## 13. Evaluate the Logistic Regression Model

In [ ]:
lr_model = trained_models['Logistic Regression']
lr_pred = lr_model.predict(X_test)
lr_proba = lr_model.predict_proba(X_test)[:, 1]

print(classification_report(y_test, lr_pred, target_names=['Stayed', 'Churned']))

cm = confusion_matrix(y_test, lr_pred)
plt.figure(figsize=(6,5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=['Stayed','Churned'], yticklabels=['Stayed','Churned'])
plt.title('Logistic Regression – Confusion Matrix')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.show()

## 14. Compare ROC Curves

In [ ]:
plt.figure(figsize=(8,6))
for name, model in trained_models.items():
    proba = model.predict_proba(X_test)[:, 1]
    RocCurveDisplay.from_predictions(y_test, proba, name=name, ax=plt.gca())
plt.title('ROC Curves – Model Comparison')
plt.show()

## 15. Random Forest Feature Importance

In [ ]:
rf_model = trained_models['Random Forest']
rf_estimator = rf_model.named_steps['model']
rf_preprocessor = rf_model.named_steps['preprocessor']

feature_names = rf_preprocessor.get_feature_names_out()
importance = pd.Series(rf_estimator.feature_importances_, index=feature_names).sort_values(ascending=False).head(15)

display(importance.to_frame('Importance').round(4))

plt.figure(figsize=(9,6))
importance.sort_values().plot(kind='barh')
plt.title('Top 15 Random Forest Feature Importances')
plt.xlabel('Importance')
plt.ylabel('Feature')
plt.show()

## 16. Example Churn-Risk Scoring

In [ ]:
risk_sample = data.drop(columns=['Churn']).sample(10, random_state=42)
risk_ids = risk_sample['customerID'].values
risk_features = risk_sample.drop(columns=['customerID'])
risk_probability = lr_model.predict_proba(risk_features)[:, 1]

risk_output = pd.DataFrame({
    'Customer ID': risk_ids,
    'Predicted Churn Probability': risk_probability
})
risk_output['Risk Category'] = pd.cut(
    risk_output['Predicted Churn Probability'],
    bins=[-0.01, 0.33, 0.66, 1.0],
    labels=['Low', 'Medium', 'High']
)
display(risk_output.sort_values('Predicted Churn Probability', ascending=False).round({'Predicted Churn Probability':3}))

## 17. Portfolio-Ready Findings

### Data Findings
- Churn is not evenly distributed across the customer base.
- Contract type and customer tenure show visible differences in churn rates.
- Monthly charges and service combinations provide useful segmentation variables.
- The EDA demonstrates why customer churn should be examined across multiple dimensions rather than with one metric.

### Machine Learning Findings
- Multiple supervised classification models were trained using the same train/test split.
- Accuracy, precision, recall, F1 score, and ROC-AUC were used together rather than relying on accuracy alone.
- The selected model should be based on the business objective and the trade-off between false positives and false negatives.
- Feature importance from the Random Forest model provides an interpretable view of which encoded variables contribute most to its predictions.

### Limitations
- The dataset is a historical sample rather than a live production dataset.
- The random train/test split is appropriate for this educational project, but a production system would require temporal validation and ongoing monitoring.
- Model predictions indicate statistical associations in the dataset and should not be interpreted as proof that a particular customer characteristic causes churn.

## 18. Conclusion

This project demonstrates an end-to-end Data Science workflow using Python: business problem definition, data loading, data-quality checks, cleaning, feature engineering, exploratory analysis, visualization, machine learning, model evaluation, feature interpretation, and risk scoring.

**Technologies:** Python, Pandas, NumPy, Matplotlib, Seaborn, Scikit-learn, Google Colab.

**Project status:** Completed as a portfolio-ready educational project for Codomax Internship Module 6.